In [2]:
import numpy as np # linear algebra
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

In [3]:
recipe = pd.read_json('./train.json')

In [4]:
recipe.head()

,id,cuisine,ingredients
0,10259,greek,"[romaine lettuce, black olives, grape tomatoes..."
1,25693,southern_us,"[plain flour, ground pepper, salt, tomatoes, g..."
2,20130,filipino,"[eggs, pepper, salt, mayonaise, cooking oil, g..."
3,22213,indian,"[water, vegetable oil, wheat, salt]"
4,13162,indian,"[black pepper, shallots, cornflour, cayenne pe..."


### How big is the dataset, columns and whether null exists?

In [5]:
print('Shape:',recipe.shape) 
print('Columns:',recipe.columns)

print('Whether Null exists:\n',recipe.isnull().sum())

Shape: (39774, 3)
Columns: Index(['id', 'cuisine', 'ingredients'], dtype='object')
Whether Null exists:
 id             0
cuisine        0
ingredients    0
dtype: int64


### How many entries are there for each cuisine?

In [6]:
recipe['cuisine'].value_counts()

cuisine
italian         7838
mexican         6438
southern_us     4320
indian          3003
chinese         2673
french          2646
cajun_creole    1546
thai            1539
japanese        1423
greek           1175
spanish          989
korean           830
vietnamese       825
moroccan         821
british          804
filipino         755
irish            667
jamaican         526
russian          489
brazilian        467
Name: count, dtype: int64

In [7]:
# bar plot for count of entries for each cuisine
x = recipe['cuisine'].value_counts().index
y = recipe['cuisine'].value_counts().values

df = pd.DataFrame({
    'Cuisine':x,
    'These many entries':y
})
#fig = sns.countplot(recipe['cuisine'])
fig = px.bar(df,
             x='Cuisine',
             y='These many entries',
             color='Cuisine')
fig.show()


### We need to find the ingredient which is used the max time in all the recipes

In [8]:
all_ingredients = [] # list to store all ingredients
for indiv_ingredient_list in recipe['ingredients'].values:
    for ingredient in indiv_ingredient_list:
        all_ingredients.append(ingredient)

In [9]:
len(all_ingredients) # 4 lacs ingredients 

428275

In [10]:
# Convert that list in a Pandas DataFrame so that we can apply value_counts
ingredients_together = pd.DataFrame(all_ingredients)

In [11]:
ingredients_together

,0
0,romaine lettuce
1,black olives
2,grape tomatoes
3,garlic
4,pepper
...,...
428270,garlic
428271,white sugar
428272,roma tomatoes
428273,celery


In [12]:
ingredients_together.value_counts()[0:30] # for first 30

0                     
salt                      18049
olive oil                  7972
onions                     7972
water                      7457
garlic                     7380
sugar                      6434
garlic cloves              6237
butter                     4848
ground black pepper        4785
all-purpose flour          4632
pepper                     4438
vegetable oil              4385
eggs                       3388
soy sauce                  3296
kosher salt                3113
green onions               3078
tomatoes                   3058
large eggs                 2948
carrots                    2814
unsalted butter            2782
extra-virgin olive oil     2747
ground cumin               2747
black pepper               2627
milk                       2263
chili powder               2036
oil                        1970
red bell pepper            1939
purple onion               1896
scallions                  1891
grated parmesan cheese     1886
Name: count, dtyp

In [13]:
ingredients_together.value_counts()

0                          
salt                           18049
olive oil                       7972
onions                          7972
water                           7457
garlic                          7380
                               ...  
dark chocolate cocoa powder        1
dark crème de cacao                1
dark lager                         1
shishito chile                     1
padron peppers                     1
Name: count, Length: 6714, dtype: int64

### Salt is used in max recipes

In [14]:
# bar plot for count of entries for each cuisine
x = ingredients_together.value_counts()[0:30].index.tolist()
y = ingredients_together.value_counts()[0:30].values

df = pd.DataFrame({
    'Ingredient':x,
    'These many entries':y
})
#fig = sns.countplot(recipe['cuisine'])
fig = px.pie(df,
             names='Ingredient',
             values='These many entries',
             color='Ingredient')
fig.show()


## Let's work on prediction

In [15]:
recipe['cuisine'].value_counts()

cuisine
italian         7838
mexican         6438
southern_us     4320
indian          3003
chinese         2673
french          2646
cajun_creole    1546
thai            1539
japanese        1423
greek           1175
spanish          989
korean           830
vietnamese       825
moroccan         821
british          804
filipino         755
irish            667
jamaican         526
russian          489
brazilian        467
Name: count, dtype: int64

### Convert column cuisine into numeric

In [16]:
recipe['cuisine'] = recipe['cuisine'].map({'italian':1,
                       'mexican':2,
                       'southern_us':3,
                       'indian':4,
                       'chinese':5,
                       'french':6,
                       'cajun_creole':7,
                       'thai':8,
                       'japanese':9,
                       'greek':10,
                       'spanish':11,
                       'korean':12,
                       'vietnamese':13,
                       'moroccan':14,
                       'british':15,
                       'filipino':16,
                       'irish':17,
                       'jamaican':18,
                       'russian':19,
                       'brazilian':20
})

### Unique ingredient list assigned 1/0

In [17]:
unique_ingredients = ingredients_together.squeeze().value_counts().index.tolist()
ingredients_keys = {ingredient: i for i, ingredient in enumerate(unique_ingredients)}
ingredients_keys

{'salt': 0,
 'olive oil': 1,
 'onions': 2,
 'water': 3,
 'garlic': 4,
 'sugar': 5,
 'garlic cloves': 6,
 'butter': 7,
 'ground black pepper': 8,
 'all-purpose flour': 9,
 'pepper': 10,
 'vegetable oil': 11,
 'eggs': 12,
 'soy sauce': 13,
 'kosher salt': 14,
 'green onions': 15,
 'tomatoes': 16,
 'large eggs': 17,
 'carrots': 18,
 'unsalted butter': 19,
 'ground cumin': 20,
 'extra-virgin olive oil': 21,
 'black pepper': 22,
 'milk': 23,
 'chili powder': 24,
 'oil': 25,
 'red bell pepper': 26,
 'purple onion': 27,
 'scallions': 28,
 'grated parmesan cheese': 29,
 'sesame oil': 30,
 'corn starch': 31,
 'ginger': 32,
 'baking powder': 33,
 'jalapeno chilies': 34,
 'dried oregano': 35,
 'chopped cilantro fresh': 36,
 'fresh lemon juice': 37,
 'diced tomatoes': 38,
 'fresh parsley': 39,
 'minced garlic': 40,
 'chicken broth': 41,
 'sour cream': 42,
 'cayenne pepper': 43,
 'fresh ginger': 44,
 'brown sugar': 45,
 'cooking spray': 46,
 'shallots': 47,
 'garlic powder': 48,
 'lime': 49,
 'lemo

In [18]:
recipe_ids = recipe.id
random_recipe_id = np.random.choice(recipe_ids, 10, replace=False)
random_recipe_id

array([ 5121, 47236, 48018, 18233, 38132,  5364,  3370, 34157, 12450,
       38009])

In [19]:
chosen = random_recipe_id[0]
chosen_ingredients = recipe[recipe['id'] == chosen]['ingredients'].values[0]
chosen_ingredients

['black pepper',
 'marinara sauce',
 'baby spinach',
 'dried oregano',
 'low-fat cottage cheese',
 'low-fat ricotta cheese',
 'garlic',
 'olive oil',
 'egg whites',
 'sea salt',
 'grated parmesan cheese',
 'whole wheat lasagna noodles',
 'shredded mozzarella cheese']

In [20]:
def ingredients_to_vector(chosen_ingredients):
    vector = np.zeros(len(ingredients_keys))
    for ingredient in chosen_ingredients:
        index = ingredients_keys.get(ingredient)
        if ingredient in unique_ingredients:
            vector[index] = 1
            # print(f"Ingredient: {ingredient}, Index: {index}, Vector: {vector[index]}")
    return vector

### Split input and output columns

In [21]:
X = np.array([ingredients_to_vector(ings) for ings in recipe['ingredients']])
Y = recipe['cuisine']
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.]])

In [22]:
Y

0        10
1         3
2        16
3         4
4         4
         ..
39769    17
39770     1
39771    17
39772     5
39773     2
Name: cuisine, Length: 39774, dtype: int64

### Create test, train, validation split

In [23]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,random_state=40,stratify=Y)
X_valid,X_test,Y_valid,Y_test = train_test_split(X_test,Y_test,test_size=0.5,random_state=40,stratify=Y_test)

### Using SVC

In [ ]:
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

valid_Acc = []
train_Acc = []
C = [0.001, 0.01, 0.1, 1, 10, 100]
for c in C:
    model = LinearSVC(C=c, max_iter=5000)
    model.fit(X_train, Y_train)
    train_Acc.append(accuracy_score(Y_train, model.predict(X_train)))
    valid_Acc.append(accuracy_score(Y_valid, model.predict(X_valid)))

plt.plot(C, train_Acc, marker="o", label="Train")
plt.plot(C, valid_Acc, marker="o", label="Validation")
plt.xscale("log")
plt.xlabel("C")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Train vs. validation accuracy across C")
plt.show()

In [1]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

final = CalibratedClassifierCV(LinearSVC(C=0.1, max_iter=5000), cv=5)
final.fit(X_train, Y_train)

pred = final.predict_proba(X_test)[0]

cuisines = final.classes_
result = sorted(zip(cuisines, pred), key=lambda x: x[1], reverse=True)

print("top cuisines:", result[:5])
# print(classification_report(Y_test, final.predict(X_test)))

NameError: name 'LinearSVC' is not defined